## Tree-Based Models — Classification (`Demand_Category`)

Same evaluation harness as `logistic_regression.ipynb`:

- Same 4 base datasets: `train`, `train_preserved_imputed`, `train_clean_imputed`, `train_clean`
- Same 5 feature-engineering stages (baseline → skew transforms → weather categories → +Weekday → cyclic time)
- Same `StratifiedKFold(5)` + macro-F1 evaluation

Model families are pulled from the regression project's `tree_models.ipynb`: **Random Forest, XGBoost, LightGBM, CatBoost**,
adapted to 3-class classification (`Demand_Category` ∈ {0 = Low, 1 = Normal, 2 = High}) instead of `log1p`-target regression.

Categorical handling follows the regression notebook's per-model conventions:
- **Random Forest** goes through the same imputer + one-hot `ColumnTransformer` used for Logistic Regression (it cannot
  consume raw NaNs or raw categoricals).
- **XGBoost / LightGBM / CatBoost** use their native categorical handling (pandas `category` dtype for XGBoost/LightGBM,
  string `cat_features` for CatBoost) — the same idiom as the regression `train_xgboost_model` / `train_lgbm_model` /
  `train_catboost_model` functions.

**Additional requirement:** XGBoost and LightGBM are *also* evaluated directly on `train_clean_nan` and
`train_preserved_nan` — real, un-imputed NaNs — since both models handle missing values natively. This checks whether
native NaN handling beats pre-imputation for this dataset. Random Forest and CatBoost are not run on these two extra
datasets (Random Forest can't handle raw NaN at all; CatBoost was not requested for this comparison).


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (required to unlock IterativeImputer)
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

df_train = pd.read_csv("train.csv")
df_train_preserved_imputed = pd.read_csv("train_preserved_imputed.csv")
df_train_clean_imputed = pd.read_csv("train_clean_imputed.csv")
df_train_clean = pd.read_csv("train_clean.csv")

# Raw-NaN datasets — used only for XGBoost / LightGBM (native missing-value support).
df_train_clean_nan = pd.read_csv("train_clean_nan.csv")
df_train_preserved_nan = pd.read_csv("train_preserved_nan.csv")


XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <1A0D8152-BF46-3BE0-B651-EE965C187777> /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/O

In [ ]:
# ============================================================
# COMMON CONFIGURATION
# ============================================================

TARGET = "Demand_Category"

DATASETS = {
    "train": df_train.copy(),
    "train_preserved_imputed": df_train_preserved_imputed.copy(),
    "train_clean_imputed": df_train_clean_imputed.copy(),
    "train_clean": df_train_clean.copy(),
}

# Only fed to XGBoost / LightGBM (see markdown above) — raw NaNs, no imputation step at all.
NAN_NATIVE_DATASETS = {
    "train_clean_nan": df_train_clean_nan.copy(),
    "train_preserved_nan": df_train_preserved_nan.copy(),
}

BASE_NUMERIC_FEATURES = [
    "Hour",
    "Temperature",
    "Humidity",
    "Wind speed",
    "Visibility",
    "Dew point temperature",
    "Solar Radiation",
    "Rainfall",
    "Snowfall",
]

BASE_CATEGORICAL_FEATURES = [
    "Seasons",
    "Holiday",
    "Functioning Day",
]

WEATHER_NUMERIC_FEATURES = [
    "Wind speed",
    "Solar Radiation",
    "Rainfall",
    "Snowfall",
    "Visibility",
]

CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

TREE_MODELS = ["RandomForest", "XGBoost", "LightGBM", "CatBoost"]
NAN_NATIVE_MODELS = ["XGBoost", "LightGBM"]


In [ ]:
def build_preprocessor(numeric_features, categorical_features, X_columns):
    """
    Preprocessing pipeline for Random Forest (identical to the one used in
    logistic_regression.ipynb).

    Numerical:
        Iterative imputation -> standardization

    Categorical:
        Most-frequent imputation -> one-hot encoding
    """

    num_cols = [
        col for col in numeric_features
        if col in X_columns
    ]

    cat_cols = [
        col for col in categorical_features
        if col in X_columns
    ]

    numerical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                IterativeImputer(
                    max_iter=10,
                    random_state=42
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop="first"
                )
            )
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numerical_transformer,
                num_cols
            ),
            (
                "cat",
                categorical_transformer,
                cat_cols
            )
        ],
        remainder="drop"
    )

    return preprocessor


In [ ]:
# ============================================================
# FEATURE ENGINEERING (identical to logistic_regression.ipynb)
# ============================================================

def add_skew_transformations(df):
    """
    Add transformed versions of skewed weather variables.
    Original variables are retained.
    """

    out = df.copy()

    right_skewed = [
        "Wind speed",
        "Solar Radiation",
        "Rainfall",
        "Snowfall"
    ]

    for col in right_skewed:
        if col in out.columns:
            # Negative values are physically invalid for these variables.
            # Turn them into NaN so the pipeline can impute them.
            values = out[col].mask(out[col] < 0)
            out[f"{col}_log1p"] = np.log1p(values)

    if "Visibility" in out.columns:
        values = out["Visibility"].mask(out["Visibility"] < 0)
        out["Visibility_pow3"] = values ** 3

    return out


def add_weather_categories(df):
    """
    Add categorical versions of heavily skewed weather variables.
    Original continuous features are retained.
    """

    out = df.copy()

    if "Visibility" in out.columns:
        bins = [-np.inf, 500, 1500, np.inf]
        out["Visibility_Cat"] = pd.cut(
            out["Visibility"], bins=bins, labels=["low", "medium", "high"]
        )

    if "Solar Radiation" in out.columns:
        bins = [-np.inf, 0.1, 1.5, np.inf]
        out["Solar_Radiation_Cat"] = pd.cut(
            out["Solar Radiation"], bins=bins, labels=["none", "low", "high"]
        )

    if "Rainfall" in out.columns:
        bins = [-np.inf, 0.1, 2.0, np.inf]
        out["Rainfall_Cat"] = pd.cut(
            out["Rainfall"], bins=bins, labels=["none", "light", "heavy"]
        )

    if "Snowfall" in out.columns:
        bins = [-np.inf, 0.1, 1.0, np.inf]
        out["Snowfall_Cat"] = pd.cut(
            out["Snowfall"], bins=bins, labels=["none", "light", "heavy"]
        )

    return out


def add_weekday(df):
    """Add weekday derived from Date. Monday = 0, Sunday = 6."""

    out = df.copy()

    if "Date" in out.columns:
        date_parsed = pd.to_datetime(out["Date"], dayfirst=True, errors="coerce")
        out["Weekday"] = date_parsed.dt.dayofweek

    return out


def add_month(df):
    out = df.copy()

    if "Date" in out.columns:
        date_parsed = pd.to_datetime(out["Date"], dayfirst=True, errors="coerce")
        out["Month"] = date_parsed.dt.month

    return out


def add_all_cyclic_time_features(df):
    """Add cyclic representations for Hour, Weekday, Month, Seasons."""

    out = df.copy()

    if "Hour" in out.columns:
        out["Hour_sin"] = np.sin(2 * np.pi * out["Hour"] / 24)
        out["Hour_cos"] = np.cos(2 * np.pi * out["Hour"] / 24)

    if "Weekday" in out.columns:
        out["Weekday_sin"] = np.sin(2 * np.pi * out["Weekday"] / 7)
        out["Weekday_cos"] = np.cos(2 * np.pi * out["Weekday"] / 7)

    if "Month" in out.columns:
        out["Month_sin"] = np.sin(2 * np.pi * (out["Month"] - 1) / 12)
        out["Month_cos"] = np.cos(2 * np.pi * (out["Month"] - 1) / 12)

    if "Seasons" in out.columns:
        season_mapping = {"Winter": 0, "Spring": 1, "Summer": 2, "Autumn": 3}
        season_num = out["Seasons"].map(season_mapping)
        out["Season_sin"] = np.sin(2 * np.pi * season_num / 4)
        out["Season_cos"] = np.cos(2 * np.pi * season_num / 4)

    return out


SKEW_NUMERIC_FEATURES = BASE_NUMERIC_FEATURES + [
    "Wind speed_log1p",
    "Solar Radiation_log1p",
    "Rainfall_log1p",
    "Snowfall_log1p",
    "Visibility_pow3",
]

WEATHER_CATEGORY_FEATURES = BASE_CATEGORICAL_FEATURES + [
    "Visibility_Cat",
    "Solar_Radiation_Cat",
    "Rainfall_Cat",
    "Snowfall_Cat",
]


In [ ]:
def evaluate_tree_model(df, numeric_features, categorical_features, model_name, experiment_name=""):
    """
    Evaluate a single tree-based classifier using the same Stratified 5-fold
    CV / macro-F1 harness as evaluate_logistic_regression().

    Categorical handling matches the regression tree_models.ipynb pattern:
      - CatBoost:            categoricals as strings, native cat_features.
      - XGBoost / LightGBM:  categoricals as pandas 'category' dtype
                              (native categorical + native NaN support).
      - RandomForest:        goes through build_preprocessor() (imputer + one-hot),
                              since sklearn's RandomForestClassifier cannot
                              consume raw NaN or raw categoricals.
    """

    feature_cols = [c for c in numeric_features + categorical_features if c in df.columns]
    cat_cols = [c for c in categorical_features if c in df.columns]

    X = df[feature_cols].copy()
    y = df[TARGET].astype(int)

    fold_scores = []

    for train_idx, val_idx in CV.split(X, y):
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        if model_name == "CatBoost":
            for col in cat_cols:
                X_train[col] = X_train[col].astype(str)
                X_val[col] = X_val[col].astype(str)

            model = CatBoostClassifier(
                random_state=42,
                verbose=False,
                loss_function="MultiClass"
            )
            model.fit(X_train, y_train, cat_features=cat_cols)
            preds = model.predict(X_val).astype(int).ravel()

        elif model_name in ("XGBoost", "LightGBM"):
            for col in cat_cols:
                X_train[col] = X_train[col].astype("category")
                X_val[col] = X_val[col].astype("category")

            if model_name == "XGBoost":
                model = XGBClassifier(
                    objective="multi:softmax",
                    num_class=3,
                    enable_categorical=True,
                    tree_method="hist",
                    random_state=42,
                    n_jobs=-1
                )
            else:
                model = LGBMClassifier(
                    random_state=42,
                    n_jobs=-1,
                    verbose=-1
                )

            model.fit(X_train, y_train)
            preds = model.predict(X_val)

        elif model_name == "RandomForest":
            preprocessor = build_preprocessor(numeric_features, categorical_features, X.columns)
            model = Pipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1)),
                ]
            )
            model.fit(X_train, y_train)
            preds = model.predict(X_val)

        else:
            raise ValueError(f"Unknown model_name: {model_name}")

        fold_scores.append(f1_score(y_val, preds, average="macro"))

    fold_scores = np.array(fold_scores)

    result = {
        "Model": model_name,
        "Experiment": experiment_name,
        "Mean Macro F1": fold_scores.mean(),
        "Std Macro F1": fold_scores.std(),
        "Fold 1": fold_scores[0],
        "Fold 2": fold_scores[1],
        "Fold 3": fold_scores[2],
        "Fold 4": fold_scores[3],
        "Fold 5": fold_scores[4],
    }

    print(
        f"[{model_name:<12}] {experiment_name:<50} "
        f"Macro-F1 = {fold_scores.mean():.4f} (± {fold_scores.std():.4f})"
    )

    return result


In [ ]:
results = []

for dataset_name, original_df in DATASETS.items():

    print("\n")
    print("=" * 90)
    print(f"DATASET: {dataset_name}")
    print("=" * 90)

    # --------------------------------------------------------
    # 1. GIVEN DATASET / BASELINE
    # --------------------------------------------------------

    df_exp = original_df.copy()

    for model_name in TREE_MODELS:
        results.append(
            evaluate_tree_model(
                df=df_exp,
                numeric_features=BASE_NUMERIC_FEATURES,
                categorical_features=BASE_CATEGORICAL_FEATURES,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 1. Baseline"
            )
        )

    # --------------------------------------------------------
    # 2. TRANSFORM SKEWED VARIABLES
    # --------------------------------------------------------

    df_exp = add_skew_transformations(df_exp)

    for model_name in TREE_MODELS:
        results.append(
            evaluate_tree_model(
                df=df_exp,
                numeric_features=SKEW_NUMERIC_FEATURES,
                categorical_features=BASE_CATEGORICAL_FEATURES,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 2. Skew transformations"
            )
        )

    # --------------------------------------------------------
    # 3. CATEGORIZE WEATHER FEATURES
    # --------------------------------------------------------

    df_exp = add_weather_categories(df_exp)

    for model_name in TREE_MODELS:
        results.append(
            evaluate_tree_model(
                df=df_exp,
                numeric_features=BASE_NUMERIC_FEATURES + ["Wind speed_log1p"],
                categorical_features=WEATHER_CATEGORY_FEATURES,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 3. + Weather categories"
            )
        )

    # --------------------------------------------------------
    # 4. ADD WEEKDAY
    # --------------------------------------------------------

    df_exp = add_weekday(df_exp)

    categorical_with_weekday = BASE_CATEGORICAL_FEATURES + ["Weekday"]

    for model_name in TREE_MODELS:
        results.append(
            evaluate_tree_model(
                df=df_exp,
                numeric_features=SKEW_NUMERIC_FEATURES,
                categorical_features=categorical_with_weekday,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 4. + Weekday"
            )
        )

    # --------------------------------------------------------
    # 5. TRANSFORM TIME VARIABLES TO CYCLIC
    # --------------------------------------------------------

    df_exp = add_weekday(df_exp)
    df_exp = add_month(df_exp)
    df_exp = add_all_cyclic_time_features(df_exp)

    numeric_cyclic = SKEW_NUMERIC_FEATURES + [
        "Weekday_sin", "Weekday_cos",
        "Month_sin", "Month_cos",
        "Hour_sin", "Hour_cos",
        "Season_sin", "Season_cos",
    ]

    categorical_cyclic = ["Holiday", "Functioning Day"]

    for model_name in TREE_MODELS:
        results.append(
            evaluate_tree_model(
                df=df_exp,
                numeric_features=numeric_cyclic,
                categorical_features=categorical_cyclic,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 5. Cyclic time: Hour+Weekday+Month+Season"
            )
        )




DATASET: train
[RandomForest] train | 1. Baseline                                Macro-F1 = 0.8014 (± 0.0149)
[XGBoost     ] train | 1. Baseline                                Macro-F1 = 0.8102 (± 0.0111)
[LightGBM    ] train | 1. Baseline                                Macro-F1 = 0.8088 (± 0.0150)
[CatBoost    ] train | 1. Baseline                                Macro-F1 = 0.8040 (± 0.0149)
[RandomForest] train | 2. Skew transformations                    Macro-F1 = 0.7967 (± 0.0117)
[XGBoost     ] train | 2. Skew transformations                    Macro-F1 = 0.8073 (± 0.0139)
[LightGBM    ] train | 2. Skew transformations                    Macro-F1 = 0.8078 (± 0.0143)
[CatBoost    ] train | 2. Skew transformations                    Macro-F1 = 0.8066 (± 0.0149)
[RandomForest] train | 3. + Weather categories                    Macro-F1 = 0.7962 (± 0.0149)
[XGBoost     ] train | 3. + Weather categories                    Macro-F1 = 0.8070 (± 0.0138)
[LightGBM    ] train | 3. + Weath

## Additional: native-NaN handling (XGBoost / LightGBM only)

`train_clean_nan` and `train_preserved_nan` still contain real, un-imputed missing values. XGBoost and LightGBM
both route missing values internally during tree construction, so they are run directly on these two datasets —
no `IterativeImputer` / `SimpleImputer` step at all — to see whether native handling beats the pre-imputed variants
above. Random Forest and CatBoost are intentionally excluded from this comparison (Random Forest cannot consume
raw NaN; CatBoost's native-NaN comparison was not requested).

Same 5 feature-engineering stages are applied for consistency with the rest of the notebook.


In [ ]:
for dataset_name, original_df in NAN_NATIVE_DATASETS.items():

    print("\n")
    print("=" * 90)
    print(f"DATASET (native NaN, XGBoost/LightGBM only): {dataset_name}")
    print("=" * 90)

    df_exp = original_df.copy()

    for model_name in NAN_NATIVE_MODELS:
        results.append(
            evaluate_tree_model(
                df=df_exp,
                numeric_features=BASE_NUMERIC_FEATURES,
                categorical_features=BASE_CATEGORICAL_FEATURES,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 1. Baseline"
            )
        )

    df_exp = add_skew_transformations(df_exp)
    for model_name in NAN_NATIVE_MODELS:
        results.append(
            evaluate_tree_model(
                df=df_exp,
                numeric_features=SKEW_NUMERIC_FEATURES,
                categorical_features=BASE_CATEGORICAL_FEATURES,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 2. Skew transformations"
            )
        )

    df_exp = add_weather_categories(df_exp)
    for model_name in NAN_NATIVE_MODELS:
        results.append(
            evaluate_tree_model(
                df=df_exp,
                numeric_features=BASE_NUMERIC_FEATURES + ["Wind speed_log1p"],
                categorical_features=WEATHER_CATEGORY_FEATURES,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 3. + Weather categories"
            )
        )

    df_exp = add_weekday(df_exp)
    categorical_with_weekday = BASE_CATEGORICAL_FEATURES + ["Weekday"]
    for model_name in NAN_NATIVE_MODELS:
        results.append(
            evaluate_tree_model(
                df=df_exp,
                numeric_features=SKEW_NUMERIC_FEATURES,
                categorical_features=categorical_with_weekday,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 4. + Weekday"
            )
        )

    df_exp = add_weekday(df_exp)
    df_exp = add_month(df_exp)
    df_exp = add_all_cyclic_time_features(df_exp)
    numeric_cyclic = SKEW_NUMERIC_FEATURES + [
        "Weekday_sin", "Weekday_cos",
        "Month_sin", "Month_cos",
        "Hour_sin", "Hour_cos",
        "Season_sin", "Season_cos",
    ]
    categorical_cyclic = ["Holiday", "Functioning Day"]
    for model_name in NAN_NATIVE_MODELS:
        results.append(
            evaluate_tree_model(
                df=df_exp,
                numeric_features=numeric_cyclic,
                categorical_features=categorical_cyclic,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 5. Cyclic time: Hour+Weekday+Month+Season"
            )
        )




DATASET (native NaN, XGBoost/LightGBM only): train_clean_nan
[XGBoost     ] train_clean_nan | 1. Baseline                      Macro-F1 = 0.8496 (± 0.0078)
[LightGBM    ] train_clean_nan | 1. Baseline                      Macro-F1 = 0.8505 (± 0.0062)
[XGBoost     ] train_clean_nan | 2. Skew transformations          Macro-F1 = 0.8496 (± 0.0078)
[LightGBM    ] train_clean_nan | 2. Skew transformations          Macro-F1 = 0.8505 (± 0.0062)
[XGBoost     ] train_clean_nan | 3. + Weather categories          Macro-F1 = 0.8487 (± 0.0040)
[LightGBM    ] train_clean_nan | 3. + Weather categories          Macro-F1 = 0.8513 (± 0.0043)
[XGBoost     ] train_clean_nan | 4. + Weekday                     Macro-F1 = 0.8833 (± 0.0093)
[LightGBM    ] train_clean_nan | 4. + Weekday                     Macro-F1 = 0.8860 (± 0.0077)
[XGBoost     ] train_clean_nan | 5. Cyclic time: Hour+Weekday+Month+Season Macro-F1 = 0.8966 (± 0.0110)
[LightGBM    ] train_clean_nan | 5. Cyclic time: Hour+Weekday+Month+Seaso

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Mean Macro F1",
    ascending=False
).reset_index(drop=True)

display(
    results_df[
        ["Model", "Experiment", "Mean Macro F1", "Std Macro F1"]
    ]
)


,Model,Experiment,Mean Macro F1,Std Macro F1
0,LightGBM,train_clean_nan | 5. Cyclic time: Hour+Weekday...,0.902811,0.006997
1,CatBoost,train_clean_imputed | 5. Cyclic time: Hour+Wee...,0.902104,0.008030
2,LightGBM,train_clean_imputed | 5. Cyclic time: Hour+Wee...,0.900930,0.006920
3,CatBoost,train_clean | 5. Cyclic time: Hour+Weekday+Mon...,0.897655,0.007815
4,XGBoost,train_clean_imputed | 5. Cyclic time: Hour+Wee...,0.896707,0.008886
...,...,...,...,...
95,CatBoost,train | 3. + Weather categories,0.804094,0.016438
96,CatBoost,train | 1. Baseline,0.804015,0.014892
97,RandomForest,train | 1. Baseline,0.801394,0.014917
98,RandomForest,train | 2. Skew transformations,0.796688,0.011689


In [ ]:
# Split "Dataset | Step" back apart with an explicit literal separator
# (regex=False avoids the bug in logistic_regression.ipynb, where
# str.split(" | ") was silently interpreted as a regex alternation
# on the bare space character and collapsed every Step to "|").
results_table = results_df.copy()

results_table["Dataset"] = results_table["Experiment"].str.split(" | ", regex=False).str[0]
results_table["Step"] = results_table["Experiment"].str.split(" | ", regex=False).str[1]

best_per_dataset_model = (
    results_table
    .loc[
        lambda df: df.groupby(["Dataset", "Model"])["Mean Macro F1"]
        .transform("max") == df["Mean Macro F1"]
    ]
    .sort_values("Mean Macro F1", ascending=False)
)

display(
    best_per_dataset_model[
        ["Dataset", "Model", "Experiment", "Mean Macro F1", "Std Macro F1"]
    ]
)


,Dataset,Model,Experiment,Mean Macro F1,Std Macro F1
0,train_clean_nan,LightGBM,train_clean_nan | 5. Cyclic time: Hour+Weekday...,0.902811,0.006997
1,train_clean_imputed,CatBoost,train_clean_imputed | 5. Cyclic time: Hour+Wee...,0.902104,0.008030
2,train_clean_imputed,LightGBM,train_clean_imputed | 5. Cyclic time: Hour+Wee...,0.900930,0.006920
3,train_clean,CatBoost,train_clean | 5. Cyclic time: Hour+Weekday+Mon...,0.897655,0.007815
4,train_clean_imputed,XGBoost,train_clean_imputed | 5. Cyclic time: Hour+Wee...,0.896707,0.008886
5,train_clean_nan,XGBoost,train_clean_nan | 5. Cyclic time: Hour+Weekday...,0.896637,0.011018
6,train_preserved_imputed,CatBoost,train_preserved_imputed | 5. Cyclic time: Hour...,0.895305,0.010449
7,train_preserved_nan,LightGBM,train_preserved_nan | 5. Cyclic time: Hour+Wee...,0.894099,0.011534
8,train_clean,XGBoost,train_clean | 5. Cyclic time: Hour+Weekday+Mon...,0.892899,0.006857
9,train_clean,LightGBM,train_clean | 5. Cyclic time: Hour+Weekday+Mon...,0.892814,0.004990


In [ ]:
best_result = results_df.iloc[0]

print("BEST EXPERIMENT")
print("=" * 50)
print("Model:", best_result["Model"])
print("Experiment:", best_result["Experiment"])
print("Mean Macro F1:", round(best_result["Mean Macro F1"], 4))
print("Std Macro F1:", round(best_result["Std Macro F1"], 4))


BEST EXPERIMENT
Model: LightGBM
Experiment: train_clean_nan | 5. Cyclic time: Hour+Weekday+Month+Season
Mean Macro F1: 0.9028
Std Macro F1: 0.007


In [ ]:
# pivot_table (not pivot) with an explicit aggfunc — robust even if the
# (Dataset, Model, Step) key were ever duplicated, unlike the plain
# .pivot() used in logistic_regression.ipynb which crashes on duplicates.
pivot = results_table.pivot_table(
    index=["Dataset", "Model"],
    columns="Step",
    values="Mean Macro F1",
    aggfunc="mean"
)

display(pivot.round(4))


Step                                  1. Baseline  2. Skew transformations  \
Dataset                 Model                                                
train                   CatBoost           0.8040                   0.8066   
                        LightGBM           0.8088                   0.8078   
                        RandomForest       0.8014                   0.7967   
                        XGBoost            0.8102                   0.8073   
train_clean             CatBoost           0.8473                   0.8497   
                        LightGBM           0.8454                   0.8454   
                        RandomForest       0.8374                   0.8371   
                        XGBoost            0.8418                   0.8418   
train_clean_imputed     CatBoost           0.8480                   0.8488   
                        LightGBM           0.8495                   0.8480   
                        RandomForest       0.8420                   0.8402   
                        XGBoost            0.8456                   0.8465   
train_clean_nan         LightGBM           0.8505                   0.8505   
                        XGBoost            0.8496                   0.8496   
train_preserved_imputed CatBoost           0.8455                   0.8445   
                        LightGBM           0.8442                   0.8449   
                        RandomForest       0.8391                   0.8355   
                        XGBoost            0.8405                   0.8435   
train_preserved_nan     LightGBM           0.8443                   0.8443   
                        XGBoost            0.8445                   0.8445   

Step                                  3. + Weather categories  4. + Weekday  \
Dataset                 Model                                                 
train                   CatBoost                       0.8041        0.8203   
                        LightGBM                       0.8059        0.8412   
                        RandomForest                   0.7962        0.8165   
                        XGBoost                        0.8070        0.8393   
train_clean             CatBoost                       0.8472        0.8658   
                        LightGBM                       0.8452        0.8783   
                        RandomForest                   0.8370        0.8569   
                        XGBoost                        0.8455        0.8806   
train_clean_imputed     CatBoost                       0.8450        0.8687   
                        LightGBM                       0.8488        0.8825   
                        RandomForest                   0.8375        0.8615   
                        XGBoost                        0.8464        0.8830   
train_clean_nan         LightGBM                       0.8513        0.8860   
                        XGBoost                        0.8487        0.8833   
train_preserved_imputed CatBoost                       0.8384        0.8633   
                        LightGBM                       0.8444        0.8768   
                        RandomForest                   0.8363        0.8578   
                        XGBoost                        0.8454        0.8797   
train_preserved_nan     LightGBM                       0.8422        0.8764   
                        XGBoost                        0.8440        0.8786   

Step                                  5. Cyclic time: Hour+Weekday+Month+Season  
Dataset                 Model                                                    
train                   CatBoost                                         0.8524  
                        LightGBM                                         0.8536  
                        RandomForest                                     0.8440  
                        XGBoost                                          0.8502  
train_clean             CatBoost                    